In [1]:
# !pip install selenium beautifulsoup4 webdriver-manager

In [14]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import StaleElementReferenceException, TimeoutException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import time
import random
import csv
from datetime import datetime
import re

In [3]:
def connect_to_naver():
    options = Options()
    options.page_load_strategy = 'eager' # 뼈대만 로드 (좋은 설정입니다 유지하세요!)
    
    # [수정] Service(ChromeDriverManager().install()) 제거
    # 그냥 괄호 비워두면 알아서 최신 드라이버 잡습니다.
    driver = webdriver.Chrome(options=options) 
    
    driver.get("https://map.naver.com/v5")
    return driver
    

In [ ]:
def search(keyword, driver):
    # 1. 메인 화면인지 확실히 하기 위해 프레임 초기화
    driver.switch_to.default_content()
    
    # 2. 검색창 찾기 (대기 포함)
    try:
        search_box = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "input.input_search"))
        )
        time.sleep(1) # 사람처럼 잠깐 텀을 줌
        
        # 3. 검색어 입력        
        search_box.click()
        time.sleep(0.2)
        
        current_val = search_box.get_attribute("value")
        while len(current_val) > 0:
            search_box.send_keys(Keys.BACK_SPACE)
            current_val = search_box.get_attribute("value") # 지워졌는지 확인
            time.sleep(0.05) # 너무 빠르면 안 먹힐 수 있음

        search_box.send_keys(f"부산 {keyword}")
        time.sleep(0.5)
        search_box.send_keys(Keys.ENTER) # 엔터 입력
    
        try:
            WebDriverWait(driver, 3).until(
                EC.presence_of_element_located((By.ID, "searchIframe"))
            )
            print(f"'{keyword}' 엔터키 검색 성공!")
            
        except TimeoutException:
            print("엔터키가 안 먹혔습니다. 검색 버튼(돋보기)을 직접 클릭합니다.")
            
            # 검색 버튼 찾아서 클릭 (CSS: button.button_search)
            try:
                search_btn = driver.find_element(By.CSS_SELECTOR, "button.button_search")
                search_btn.click()
            except:
                # 혹시 버튼 못 찾으면 다시 엔터 시도
                search_box.send_keys(Keys.ENTER)
            
            # 클릭 후 다시 대기 (이번엔 10초까지 기다려줌)
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.ID, "searchIframe"))
            )
            print("검색 버튼 클릭으로 로딩 성공!")

    except Exception as e:
        print(f"ERROR: {e}")

In [ ]:
import time
import pandas as pd # 데이터를 엑셀처럼 다루기 위해
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains

def find_names_by_any_class(driver):
    candidates = [
        ".YwYLL",          # 호텔, 모텔
        ".moQ_p"           # 게스트하우스, 펜션
    ]
    
    for selector in candidates:
        items = driver.find_elements(By.CSS_SELECTOR, selector)
        if len(items) > 0:
            return items
            
    return []

def scroll(driver):
    try:
        driver.switch_to.default_content()
        WebDriverWait(driver, 20).until(
            EC.frame_to_be_available_and_switch_to_it((By.ID, "searchIframe"))
        )
        
        scroll_div = WebDriverWait(driver, 30).until(
            EC.presence_of_element_located((By.ID, "_pcmap_list_scroll_container"))
        )
        
        body = driver.find_element(By.TAG_NAME, "body")
        attempt = 0      # 시도 횟수
        prev_count = 0   # 이전 개수 (멈춤 감지용)
        same_count_loop = 0 # 개수가 안 늘어나고 몇 번 반복했나 체크
        
        while True:
            attempt += 1
            
            driver.execute_script("arguments[0].scrollTop = arguments[0].scrollHeight", scroll_div)
            time.sleep(10)
            
            try:
                driver.execute_script("arguments[0].click();", scroll_div)
            except:
                pass

            body.send_keys(Keys.END)
            time.sleep(10) 
            
            items = find_names_by_any_class(driver)
            current_count = len(items)
            
            print(f"스크롤 {attempt}회차: 현재 {current_count}개 확보 중...")

            if current_count >= 50:
                print("🎉 최대 개수(50개) 도달! 스크롤 종료.")
                break
            
            if attempt >= 20:
                print("⚠ 최대 시도 횟수 초과. 현재 개수에서 만족하고 종료합니다.")
                break

            if current_count == prev_count:
                same_count_loop += 1
                if same_count_loop >= 10:
                    if current_count >= 40:
                        print(f"✋ 40개 넘김({current_count}개). 더 이상 로딩 안 되므로 종료.")
                        break
                    else:
                        print("⏳ 로딩이 걸렸습니다. 조금 더 기다리고 다시 시도합니다...")
                        time.sleep(30) # 3초 더 기다려줌
            else:
                same_count_loop = 0
            
            prev_count = current_count
                
    except Exception as e:
        print(f"스크롤 중 에러 발생: {e}")

def get_page_data(driver):
    names = driver.find_elements(By.CSS_SELECTOR, ".YwYLL") # 모텔, 호텔
    if not names:
        names = driver.find_elements(By.CSS_SELECTOR, ".moQ_p") # 펜션, 게스트하우스

    name_list = []
    for name in names:
        name_list.append(name.text)
    
    return name_list

def move_to_next_page(driver):
    driver.switch_to.default_content()
    driver.switch_to.frame("searchIframe")
    
    try:
        next_btns = driver.find_elements(By.CSS_SELECTOR, ".zRM9F a") 
        if not next_btns:
            next_btn = driver.find_element(By.XPATH, "//a[contains(., '다음페이지')]")
        else:
            next_btn = next_btns[-1]

        if next_btn.get_attribute("aria-disabled") == "true":
            return False

        next_btn.click()
        time.sleep(3) # 페이지 로딩 대기
        return True

    except:
        return False


In [8]:
keys = ["펜션", "호텔", "게스트하우스", "모텔"]
final_results = [] # 모든 데이터를 담을 리스트
    
# 1. naver map 연결
driver = connect_to_naver()

for key in keys:
    result = []
    # 2. 키워드 검색
    search(key, driver)
    print(f"\n--- '{key}' 검색 시작 ---")
    
    # 3. 스크롤 및 데이터 수집
    page_count = 1
    while True:
        print(f"=== {page_count} 페이지 수집 시작 ===")
        
        # 1. 스크롤 끝까지 내리기
        scroll(driver)
        
        # 2. 데이터 긁기
        page_names = get_page_data(driver)
        print(f"-> 수집된 가게 수: {len(page_names)}개")
        
        # 결과 리스트에 추가
        result.extend(page_names)
        
        # 3. 다음 페이지 이동
        if not move_to_next_page(driver):
            print("마지막 페이지 도달! 수집 종료.")
            break
            
        page_count += 1

    print(f"{key} 총 수집된 숙박 개수: {len(result)}")
    print(result[:10]) # 상위 10개만 출력해보기
    
    final_results.extend(result)
    print(f"총 누적된 숙박 개수: {len(final_results)}개")
    # 종료

driver.quit()
# 4. 숙박명 리스트 저장
print("\ncsv 파일 저장 중...")

df = pd.DataFrame(final_results)

# 중복 제거
df = df.drop_duplicates()

file_name = "busan_accommodation_list.csv"
df.to_csv(file_name, index=False, encoding="utf-8-sig")

print(f"저장 완료! 파일명: {file_name}")
print(f"최종 수집된 가게 수: {len(df)}개")

엔터키가 안 먹혔습니다. 검색 버튼(돋보기)을 직접 클릭합니다.
검색 버튼 클릭으로 로딩 성공!

--- '펜션' 검색 시작 ---
=== 1 페이지 수집 시작 ===
스크롤 1회차: 현재 10개 확보 중...
스크롤 2회차: 현재 10개 확보 중...
스크롤 3회차: 현재 42개 확보 중...
스크롤 4회차: 현재 42개 확보 중...
스크롤 5회차: 현재 42개 확보 중...
스크롤 6회차: 현재 42개 확보 중...
스크롤 7회차: 현재 42개 확보 중...
스크롤 8회차: 현재 42개 확보 중...
스크롤 9회차: 현재 42개 확보 중...
스크롤 10회차: 현재 42개 확보 중...
스크롤 11회차: 현재 42개 확보 중...
스크롤 12회차: 현재 42개 확보 중...
스크롤 13회차: 현재 42개 확보 중...
✋ 40개 넘김(42개). 더 이상 로딩 안 되므로 종료.
-> 수집된 가게 수: 42개
=== 2 페이지 수집 시작 ===
스크롤 1회차: 현재 42개 확보 중...
스크롤 2회차: 현재 42개 확보 중...
스크롤 3회차: 현재 42개 확보 중...
스크롤 4회차: 현재 42개 확보 중...
스크롤 5회차: 현재 42개 확보 중...
스크롤 6회차: 현재 42개 확보 중...
스크롤 7회차: 현재 42개 확보 중...
스크롤 8회차: 현재 42개 확보 중...
스크롤 9회차: 현재 42개 확보 중...
스크롤 10회차: 현재 42개 확보 중...
스크롤 11회차: 현재 42개 확보 중...
✋ 40개 넘김(42개). 더 이상 로딩 안 되므로 종료.
-> 수집된 가게 수: 42개
=== 3 페이지 수집 시작 ===
스크롤 1회차: 현재 42개 확보 중...
스크롤 2회차: 현재 42개 확보 중...
스크롤 3회차: 현재 42개 확보 중...
스크롤 4회차: 현재 42개 확보 중...
스크롤 5회차: 현재 42개 확보 중...
스크롤 6회차: 현재 42개 확보 중...
스크롤 7회차: 현재 42개 확보 중...
스크롤 8회차: 현재 42개 

In [48]:
input_file = "busan_accommodation_list.csv"
output_file = "busan_accommodation_details.csv"

df = pd.read_csv(input_file)
print(f"총 {len(df)}개 숙소 로드 완료")

총 672개 숙소 로드 완료


In [65]:
# kakao map crawling
options = Options()
options.page_load_strategy = 'eager' # 뼈대만 로드 (좋은 설정입니다 유지하세요!)

driver = webdriver.Chrome(options=options) 
url = "https://map.kakao.com/"
driver.get(url)

searchloc = '부산 숙박'

search_area = driver.find_element(
    By.CSS_SELECTOR,
    "input[type='text']"
)
search_area.send_keys(searchloc)  # 검색어 전달
search_area.send_keys(Keys.ENTER)

time.sleep(5)

driver.find_element(By.XPATH, r'//*[@id="info.main.options"]/li[2]/a').send_keys(Keys.ENTER)

html = driver.page_source
soup = BeautifulSoup(html, 'html.parser') # 모든 정보 가져오기

room_lists = soup.select('.placelist > .PlaceItem')
print(f"총 {len(room_lists)}개의 숙소 정보 수집 완료")
room_lists[:3]

총 15개의 숙소 정보 수집 완료


[<li class="PlaceItem clickArea">
 <div class="utile_item" data-id="btnsWrap">
 <span class="screen_out" data-id="screenOutName">롯데호텔 부산</span>
 <a class="fav" data-id="fav" href="#none">
 <span class="ico_fav">즐겨찾기</span><span class="num"></span>
 </a>
 <a class="roadview" data-id="roadview" href="#none">로드뷰</a>
 <button class="DirectionItemInfoBtn" data-id="routeBtn" type="button">길찾기</button>
 <div class="FlagItemInfoLayer HIDDEN">
 <div class="origin" data-id="origin">출발</div>
 <div class="via" data-id="via">경유</div>
 <div class="dest" data-id="dest">도착</div>
 </div></div>
 <div class="thumb_item HIDDEN">
 <img alt="" class="thumb_g HIDDEN" data-id="image" height="70" src="" width="70"/>
 </div>
 <div class="head_item clickArea">
 <strong class="tit_name">
 <a class="link_order clickArea" data-id="icon" href="#none">A</a>
 <a class="link_name" data-id="name" href="#none" title="롯데호텔 부산">롯데호텔 부산</a>
 </strong>
 <span class="subcategory clickable" data-id="subcategory">호텔</span>
 </d

In [164]:
options = Options()
options.page_load_strategy = 'eager'

driver = webdriver.Chrome(options=options) 
detail_href = "https://place.map.kakao.com/26435197"
driver.get(detail_href)



In [ ]:
# 테그 가져오기


html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')

# 한줄평
tag = soup.select_one('.desc_boxintro')
simple_desc = tag.get_text(strip=True) if tag else ""

print("한줄평은 다음과 같다: ", simple_desc)

# 태그
def extract_tags(soup):
    tags = []
    for el in soup.find_all(['span', 'a']):
        text = el.get_text(strip=True)
        if text.startswith('#') and len(text) < 20:
            tags.append(text)
    return list(dict.fromkeys(tags))  # 중복 제거

tags = extract_tags(soup)
print("태그는 다음과 같다: ", tags)

# 영업시간
opening_hour_tag = soup.select_one('.add_mdot')
if opening_hour_tag:
    opening_hour = opening_hour_tag.get_text(strip=True)
else:
    opening_hour = None

print("영업시간:", opening_hour)

# 예약 가져오기
booking_list = soup.select('.section_booking .list_booking > li')
print(f"총 {len(booking_list)}개의 예약 정보 수집 완료")
for _, room in enumerate(booking_list):
    room_type = room.select_one('.tit_item').get_text(strip=True)
    price = room.select_one('.info_item2').get_text(strip=True)
    print(room_type, price)
    
# 리뷰 갯수
number_of_reviews = soup.select('.info_num')[0].get_text(strip=True) if soup.select('.info_num') else None
print(number_of_reviews)

# 카테고리 모텔 호텔 종류
raw_text = soup.select('.info_cate')[0]
category = raw_text.get_text(strip=True) if raw_text else None
category = category.replace("장소 카테고리", "").strip() if category else None

print("카테고리는 다음과 같다: ", category)

time.sleep(3)


한줄평은 다음과 같다:  
태그는 다음과 같다:  ['#2인실', '#MT펜션', '#가족실', '#단체민박', '#단체펜션', '#물놀이', '#바베큐장', '#산책로', '#오션뷰', '#온돌룸']
영업시간: 24시간 영업
총 4개의 예약 정보 수집 완료
룸타입5호실 45,000원~
룸타입11호실 45,000원~
룸타입8호실 50,000원~
룸타입7호실 50,000원~
7개
펜션
카테고리는 다음과 같다:  펜션


In [ ]:
#  리뷰 가져오기 제일 최신 순으로 3개
# 1. 리뷰탭
driver.execute_script("location.hash = '#review'")
time.sleep(3)
                    
def click_latest_sort(driver):
    buttons = driver.find_elements(By.CSS_SELECTOR, "a.link_sort")

    for btn in buttons:
        text = driver.execute_script(
            "return arguments[0].textContent;", btn
        ).strip()

        if "최신" in text:
            if btn.find_elements(By.CSS_SELECTOR, ".ico_chk"):
                return True

            driver.execute_script(
                "arguments[0].scrollIntoView({block:'center'});", btn
            )
            time.sleep(0.3)

            driver.execute_script("arguments[0].click();", btn)
            time.sleep(2)

            return True

    return False

# 최신순
click_latest_sort(driver)
        
# 더보기 버튼 누르기
def load_reviews_with_more(driver):
    try:
        more_buttons = driver.find_elements(By.CSS_SELECTOR, ".btn_more")
        for btn in more_buttons:
            if btn.is_displayed():
                driver.execute_script("arguments[0].click();", btn)
                time.sleep(0.8)
    except:
        pass 

load_reviews_with_more(driver)
soup = BeautifulSoup(driver.page_source, "html.parser")
review_list = soup.select('.list_review .inner_review') # list 가져오기

print(f"총 {len(review_list)}개의 리뷰 정보 수집 완료")

reviews = []
review_count = 0
for r in review_list:
    date = r.select('.txt_date')[0].text if r.select('.txt_date') else None
    review_date = datetime.strptime(date, "%Y.%m.%d.")
    
    if review_date.year == 2025 and review_count < 3:
        describe = r.select('.desc_review')[0].text if r.select('.desc_review') else None  
        if describe is not None:
            review_count += 1
            describe = describe.replace('접기', '')
            describe = describe.replace('\n', ' ')
            describe = re.sub(r'[^가-힣a-zA-Z0-9\s.,!?]', '', describe)

            reviews.append(describe)
        else:
            pass
    elif review_date.year > 2025:
        pass
    elif review_count >= 3:
        break
    else:
        break

In [67]:
for i, room in enumerate(room_lists):
        temp = []
        
        name = room.select('.head_item > .tit_name > .link_name')[0].text # 숙박명
        score = room.select('.rating > .score > em')[0].text # 별점
        addr = room.select('.addr > p')[0].text # 주소
        category = room.select('.subcategory')[0].text # 호텔, 펜션?
        phone_number = room.select('.phone')[0].text # 전화번호
        
        a_tag = room.select_one('.moreview')
        detail_href = a_tag.get('href') if a_tag else None
        print(name, score, addr, category, phone_number, detail_href)
        extract_details(detail_href)
        

롯데호텔 부산 4.3 부산 부산진구 가야대로 772 호텔 051-810-1000 https://place.map.kakao.com/7862727
한줄평은 다음과 같다:  
태그는 다음과 같다:  []
시그니엘 부산 4.1 부산 해운대구 달맞이길 30 호텔 051-922-1000 https://place.map.kakao.com/327520717
한줄평은 다음과 같다:  
태그는 다음과 같다:  []
신라스테이 해운대 3.6 부산 해운대구 해운대로570번길 46 호텔 051-912-9000 https://place.map.kakao.com/1295735325
한줄평은 다음과 같다:  
태그는 다음과 같다:  []
파크하얏트 부산 4.4 부산 해운대구 마린시티1로 51 호텔 051-990-1234 https://place.map.kakao.com/21025068


KeyboardInterrupt: 

In [ ]:
def extract_tags(soup):
    tags = []
    for el in soup.find_all(['span', 'a']):
        text = el.get_text(strip=True)
        if text.startswith('#') and len(text) < 20:
            tags.append(text)
    return list(dict.fromkeys(tags))  # 중복 제거

# 최신 순
def click_latest_sort(driver):
    buttons = driver.find_elements(By.CSS_SELECTOR, "a.link_sort")

    for btn in buttons:
        text = driver.execute_script(
            "return arguments[0].textContent;", btn
        ).strip()

        if "최신" in text:
            if btn.find_elements(By.CSS_SELECTOR, ".ico_chk"):
                return True

            driver.execute_script(
                "arguments[0].scrollIntoView({block:'center'});", btn
            )
            time.sleep(0.3)

            driver.execute_script("arguments[0].click();", btn)
            time.sleep(2)

            return True

    return False

def extract_details(detail_href):
    info = {}
    
    # 메인 페이지
    main_window = driver.current_window_handle
    time.sleep(2)
    
    driver.execute_script("window.open(arguments[0]);", detail_href)
    driver.switch_to.window(driver.window_handles[-1])
    
    print("상세 페이지로 이동 중...", driver.current_url)
    
    # 로딩 대기
    time.sleep(4)
    
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    
    # basic info
    name = soup.select('.head_item > .tit_name > .link_name')[0].text # 숙박명
    score = soup.select('.rating > .score > em')[0].text # 별점
    addr = soup.select('.addr > p')[0].text # 주소
    category = soup.select('.subcategory')[0].text # 호텔, 펜션?
    phone_number = soup.select('.phone')[0].text # 전화번호
    print("숙박명:", name)
    print("별점:", score)
    print("주소:", addr)
    print("카테고리:", category)
    print("전화번호:", phone_number)
    info['숙박명'] = name
    info['별점'] = score
    info['주소'] = addr
    info['카테고리'] = category
    info['전화번호'] = phone_number

    # 한줄평
    tag = soup.select_one('.desc_boxintro')
    simple_desc = tag.get_text(strip=True) if tag else ""
    print("한줄평은 다음과 같다: ", simple_desc)
    info['한줄평'] = simple_desc
    
    # 태그
    tags = extract_tags(soup)
    print("태그: ", tags)
    info['태그'] = tags
    
    # 영업시간
    opening_hour_tag = soup.select_one('.add_mdot')
    if opening_hour_tag:
        opening_hour = opening_hour_tag.get_text(strip=True)
    else:
        opening_hour = None
    print("영업시간:", opening_hour)
    info['영업시간'] = opening_hour

    # 가격 정보 가져오기
    booking_list = soup.select('.section_booking .list_booking > li')
    print(f"총 {len(booking_list)}개의 가격 정보 수집 완료")
    for _, room in enumerate(booking_list):
        room_type = room.select_one('.tit_item').get_text(strip=True)
        price = room.select_one('.info_item2').get_text(strip=True)
        print(room_type, price)
    
    # 리뷰 갯수
    number_of_reviews = soup.select('.info_num')[0].get_text(strip=True) if soup.select('.info_num') else None
    print("리뷰 갯수: ", number_of_reviews)
    info['리뷰 갯수'] = number_of_reviews


    # 다시 검색 탭으로 전환
    driver.close()
    driver.switch_to.window(driver.window_handles[0])
    time.sleep(2)
    return info

extract_details(detail_href)

상세 페이지로 이동 중... https://place.map.kakao.com/10928789
한줄평은 다음과 같다:  탁 트인 바다와 화려한 다이아몬드 브릿지 야경과 함께 배가되는 한화리조트 해운대
태그는 다음과 같다:  ['#가족실', '#가족호텔', '#골프텔', '#사우나', '#오션뷰', '#온돌룸', '#워크샵', '#조식서비스', '#카라반']


In [ ]:
def roomNamePrint():
    time.sleep(0.2)
    
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser') # 모든 정보 가져오기
    
    room_lists = soup.select('.placelist > .PlaceItem') # 장소 정보를 모두 가져옴
    for i, room in enumerate(room_lists):
        temp = []
        
        name = room.select('.head_item > .tit_name > .link_name')[0].text
        score = room.select('.rating > .score > em')[0].text
        addr = room.select('.addr > p')[0].text # 부산으로 시작 안하면 pass
        
        # 상세정보 탭으로 이동
        driver.find_element(By.XPATH, r'//*[@id="info.search.place.list"]/li['+str(i+1)+']/div[5]/div[4]/a[1]').send_keys(Keys.ENTER)
        driver.switch_to.window(driver.window_handles[-1])
        time.sleep(2)
        a_tag = room.select_one('.moreview')
        detail_href = a_tag.get('href') if a_tag else None
        rev = extract_details(detail_href)  # 리뷰 추출 함수 실행
        
        
        # 하나의 리스트로 만들어 room_list에 추가
        temp.append(name)
        temp.append(score)
        temp.append(addr[3:])
        temp.append(rev)
        
        results.append(temp)

In [ ]:
page = 1  # 현재 페이지
page2 = 1  # 5개 중 몇번째인지(버튼이 5개씩있어서 6번째가 되면 다시 1로 바꿔줘야함)

for i in range(1, 9):
    try:
        page2 += 1
        print(page, 'page')
        
        # 페이지 버튼 번호(1에서 5 사이 값)
        if i > 5:
            xpath = '/html/body/div[5]/div[2]/div[1]/div[7]/div[6]/div/a['+str(i-5)+']'
        else:
            xpath = '/html/body/div[5]/div[2]/div[1]/div[7]/div[6]/div/a['+str(i)+']'
        
        driver.find_element(By.XPATH, xpath).send_keys(Keys.ENTER)  # 페이지 선택
        roomNamePrint()  # 숙소 정보 크롤링
        
        
        # page2가 5를 넘어가면 다시 1로 바꿔주고 다음 버튼 클릭
        if page2 > 5:
            page2 = 1
            driver.find_element(By.XPATH, r'//* [@id="info.search.page.next"]').send_keys(Keys.ENTER)
        
        page += 1
    
    except:
        break
        
print('크롤링 완료')

1 page
크롤링 완료
